In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, TimestampType
)
from datetime import datetime, timezone
import uuid


def write_pipeline_log(
    pipeline_name,
    source_table,
    target_table,
    layer_name,
    start_timestamp,
    execution_status,
    source_count=0,
    success_count=0,
    rejected_count=0,
    duplicate_count=0,
    error_count=0,
    error_message=None
):

    end_timestamp = datetime.now(timezone.utc)

    duration_seconds = (
        end_timestamp - start_timestamp
    ).total_seconds()

    run_id = str(uuid.uuid4())

    log_data = [(
        run_id,
        pipeline_name,
        source_table,
        target_table,
        layer_name,
        start_timestamp,
        end_timestamp,
        float(duration_seconds),
        execution_status,
        int(source_count),
        int(success_count),
        int(rejected_count),
        int(duplicate_count),
        int(error_count),
        error_message,
        end_timestamp
    )]

    log_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("layer_name", StringType(), True),
        StructField("start_timestamp", TimestampType(), True),
        StructField("end_timestamp", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True),
        StructField("execution_status", StringType(), True),
        StructField("source_count", LongType(), True),
        StructField("success_count", LongType(), True),
        StructField("rejected_count", LongType(), True),
        StructField("duplicate_count", LongType(), True),
        StructField("error_count", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("created_timestamp", TimestampType(), True)
    ])

    log_df = spark.createDataFrame(
        log_data,
        log_schema
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "retail360.silver.pipeline_execution_log"
        )
    )

    return run_id